# 10 SPI (p148~p159)
- [spi yt](https://www.youtube.com/watch?v=0nVNwozXsIc)
- unlike i2c , in spi when you send a byte you receive a byte as well
- spi can be duplex or half duplex 
- while i2c can only half duplex 

# 10.1 Introduction
This Serial interface peripheral supports the following features:
- Implements a 3 wire serial protocol, variously called Serial Peripheral Interface (SPI) or Synchronous Serial Protocol (SSP).
- Implements a 2 wire version of SPI that uses a single wire as a bidirectional data wire instead of one for each direction as in standard SPI.
- Implements a LoSSI Master (Low Speed Serial Interface)
- Provides support for polled, interrupt or DMA operation.
# 10.2 SPI Master Mode 
## 10.2.1 Standard mode 
In Standard **SPI master mode** the peripheral implements the standard **3 wire serial protocol** described below.
- ?? master miso 應該是 slave mosi吧???

- Figure 10-1 SPI Master Typical Usage 
```
 ____
|    |--SPI_MOSI-------------------------------
|SPI |--SPI_MISO--------    \ ------------\  \
|____|--SPI_SCLK---\    \    |----------\  |  |  
  | |              _|___|____|__      __|__|__|__
  | |SPI_CE[0]    |clk MISO MOSI|    |          |
  | |             |   Slave 1   |    |  SLAVE 2 |
  | \-------------|_CE__________|    |_CE_______|
  | SPI_CE[1]                         |
  \------------------------------------
```

- Figure 10-2 SPI Cycle plot 
  - (chip select=chip enable(CE)) 
  - CE 一開始要是 high , 所以我們在 spi.c/spi_init 先把 pin 8 (CE) 拉高
    - main.c 傳送 spi 之前 要先 拉低 pin 8 , 傳送結束之後再拉高 pin 8 (CE)
```
         ____ (chip select=chip enable(CE))                  ______
SPI_CE       |______________________________________________|
               _   _   _   _   _   _   _   _   _         _               
SPI_SCLK _____| |_| |_| |_| |_| |_| |_| |_| |_| |_....._| |________            

SPI_MOSI ____|D7|D6 |D5 |D4 |D3 |D2 |D1 |D0 |D7 | ... |D0 |_________  

SPI_MISO ___________________________________|D7 | ... |D0 |_________
```

- Notes 
  - Slave enables itself onto SPI_MISO only when it is outputting data. At other times, output is tristate (1/0/z)
  - Different SCLK polarity and phase values are possible. Case illustrated is CPOL=0,CPHA=0
  - Data is trasmitted MSB first 
  - Transactions can be from a single byte to hundreds of bytes    



- Figure 10-3 Different Clock Polarity/phase

```
SPI_MOSI
SPI_MISO  ___|D7 |D6 |D5 |D4 |D3 |D2 |D1 |D0 |____
                _   _   _   _   _   _   _   _           
SPI_SCLK  _____| |_| |_| |_| |_| |_| |_| |_| |_ __ (a) CPHA = 0, CPOL = 0   
          _____   _   _   _   _   _   _   _   ____        
               |_| |_| |_| |_| |_| |_| |_| |_|     (b) CPHA = 0, CPOL = 1  
              _   _   _   _   _   _   _   _      
         ____| |_| |_| |_| |_| |_| |_| |_| |_____  (c) CPHA = 1, CPOL = 0   
         ____   _   _   _   _   _   _   _   ____ 
             |_| |_| |_| |_| |_| |_| |_| |_|       (d) CPHA = 1, CPOL = 1  
```


## 10.2.2 Bidirectional mode 
todo 
# 10.3 LoSSI mode 
- todo 
# 10.4 Block Diagram  
- todo 

# 10.5 SPI Register Map  (p152)
The BCM2835 devices has only one SPI interface of this type. It is referred to in all the documentation as SPI0. It has two additional mini SPI interfaces (SPI1 and SPI2). The specifiation of those can be found under 2.3 Universal SPI Master (2x). 
- The base address of this SPI0 interface is **0x7E204000**.
  - 所以 include/peripherals/spi.h 的offset is 0x00204000

## SPI Address Map (對應 spi.h/Spi0Regs)

|Address Offset|Register Name|Description                  |Size|
|--------------|-------------|-----------------------------|----|
|0x0           |CS           |SPI Master Control and status|32  |
|0x4           |FIFO         |SPI Master TX and RX FIFOS   |32  |
|0x8           |CLK          |SPI Master Clock Divider     |32  |
|0xc           |DLEN         |SPI Master Data Length       |32  |
|0x10          |LTOH         |SPI LOSSI mode TOH           |32  |
|0x14          |DC           |SPI DMA DREQ Controls        |32  |


### SPI CS register  (p153)
- Synopsis: This register contains the main control and status bits for the SPI, CS (cable select)
  - in spi.h 

|Bit(s)|Field Name|Description                        |Type|Reset|
|------|----------|-----------------------------------|----|-----|
|31:26 |          |Reserved 0                         |    |     |
|25    |LEN_LONG  |Enable long data word in Lossi mode|RW  |0x0  |
|24    |DMA_LEN   |Enable DMA mode in Lossi mode      |RW  |0x0  |
|23    |CSPOL2    |CS2 polarity:0->active low,1 high  |RW  |0x0  |
|22    |CSPOL1    |CS1 polarity:0->active low,1 high  |RW  |0x0  |
|21    |CSPOL0    |CS0 polarity:0->active low,1 high  |RW  |0x0  |
|20    |RXF       |RX FIFO FULL                       |RO  |0x0  |
|19    |RXR       |RX FIFO needs reading (full)       |RO  |0x0  |
|18    |TXD	      |**TX FIFO can accept data**        |RO  |0x1  |
|17    |RXD	      |**RX FIFO contains data**          |RO  |0x0  |
|16    |DONE      |**transfer Done**                  |RO  |0x0  |
|15    |TE_EN     |Unused                             |RW  |0x0  |
|14    |LMONO     |Unused                             |RW  |0x0  |
|13    |LEN	      |LoSSI enable                       |RW  |0x0  |
|12    |REN	      |Read Enable (if bidirectional mode)|RW  |0x1  |
|11    |ADCS      |if 1 auto deassert CS              |RW  |0x0  |
|10    |INTR      |Interrupt on RXR                   |RW  |0x0  |
|9     |INTD      |Interrupt on Done                  |RW  |0x0  |
|8     |DMAEN     |DMA enable                         |RW  |0x0  |
|7     |TA        |**Transfer Active**. assume CSPOL=0|RW  |0x0  |
|      |         |0=Transfer not active, CS lines all high||     |
|      |          |1=Transfer active, write to SPI FIFO|   |     |
|      |          |write data to Tx FIFO. TA is cleared|   |     |
|6     |CSPOL     |chip select polarity               |RW  |0x0  |
|5     |CLEAR_RX  |**Rx fifo clear**                  |RW  |0x0  |
|4     |CLEAR_TX  |**Tx fifo clear**                  |RW  |0x0  |
|3     |CPOL      |Clock polarity                     |RW  |0x0  |
|2     |CPHA      |Clock phase                        |RW  |0x0  |
|1:0   |CS        |Chip Select: 00=chip select 0      |RW  |0x0  |
|      |          |01 =chip select 1                  |   |     |
|      |          |10 =chip select 2                  |   |     |
|      |          |11 =reserved                       |   |     |



### SPI FIFO register  (p155)
- Synopsis: This register allows TX data to be written to the TX FIFO and RX data to be read from
the RX FIFO.

|Bit(s)|Field Name|Description                                      |Type|Reset|
|------|----------|-------------------------------------------------|----|-----|
|31:0  |DATA      |reads and writes will be taken as four-byte data |RW  |0x0  |
|      |          |words to be read/written to the FIFOs            |    |     |
|      |          |Poll/Interrupt Mode                              |    |     |
|      |          |Writes to the register write bytes to TX FIFO    |    |     |
|      |          |Reads from register read bytes from the RX   FIFO|    |     |